In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import sys
sys.path.insert(0, '..')
from config import TECHNOLOGIES, DATA_DIR

In [ ]:
df = pd.read_csv(DATA_DIR / "jobs.csv")
print(f"Number of vacancies: {len(df)}")
df.head(3)

In [ ]:
df = df.drop_duplicates(subset=['url']).copy()
df['title'] = df['title'].fillna('').str.lower()
df['description'] = df['description'].fillna('').str.lower()
df['text'] = df['title'] + " " + df['description']

is_senior = df['text'].str.contains(
    r'sr\.?|lead|architect|expert|senior(?:\s*level)?|(?:\d|10)\+',
    regex=True, na=False
)
is_middle = df['text'].str.contains(
    r'mid(?:dle)?|intermediate|(?:middle\s*)?level|[2-4]\+',
    regex=True, na=False
)
is_junior = df['text'].str.contains(
    r'jr\.?|intern|trainee|entry|no|without\s*exp',
    regex=True, na=False
)
df['grade'] = 'Not Specified'
df.loc[is_senior, 'grade'] = 'Senior'
df.loc[is_middle & ~is_senior, 'grade'] = 'Middle'
df.loc[is_junior & ~is_middle & ~is_senior, 'grade'] = 'Junior'
print(df['grade'].value_counts())

In [ ]:
TECH_MAP = {
    'PostgreSQL': r'postgres(?:ql)?',
    'AWS': r'aws|amazon\s*web\s*services',
    'Kubernetes': r'kubernetes|k8s',
    'REST': r'rest(?:ful)?',
    'NoSQL': r'nosql|mongo(?:db)?|cassandra',
    'React': r'react(?:js)?',
    'Vue': r'vue(?:\.js)?',
    'CI/CD': r'ci/cd|ci|cd|jenkins|git(?:lab|hub)\s*(?:ci|actions)',
    'JavaScript': r'javascript|js',
    'Machine Learning': r'machine\s*learning|ml',
    'LLM': r'llm|large\s*language\s*model|gpt'
}

for tech in TECHNOLOGIES:
    pat = TECH_MAP.get(tech, tech.lower())
    df[tech] = df['text'].str.contains(rf'\b(?:{pat})\b', regex=True, na=False)

by_grade = df.groupby('grade')[TECHNOLOGIES].sum().T
by_grade['Total'] = by_grade.sum(axis=1)
by_grade = by_grade.sort_values('Total', ascending=False)

by_grade.head(10)

In [ ]:
grade_counts = df['grade'].value_counts()
by_grade_pct = by_grade.drop(columns='Total').div(grade_counts, axis=1) * 100
by_grade_pct.head(10).round(1)

In [ ]:
grade_counts = df['grade'].value_counts()
by_grade_pct = by_grade.drop(columns='Total').div(grade_counts, axis=1) * 100

top_10 = by_grade_pct.head(10)
ax = top_10.plot(kind='bar', figsize=(14, 6), width=0.8)

plt.title('Top 10 Technologies by Job Grade (%)', fontsize=15)
plt.ylabel('Market Share (%)')
plt.xticks(rotation=45)
plt.grid(axis='y', linestyle='--', alpha=0.5)

for p in ax.patches:
    if p.get_height() > 0:
        ax.annotate(f'{p.get_height():.0f}%', 
                    (p.get_x() + p.get_width()/2, p.get_height()), 
                    ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.savefig('tech_analysis.png')

In [ ]:
total_vacancies = len(df)
top_tech = (by_grade['Total'] / total_vacancies * 100).sort_values(ascending=True)
top_tech = top_tech.tail(15)

In [ ]:
plt.figure(figsize=(10, 8))
colors = plt.cm.viridis(np.linspace(0.3, 0.8, len(top_tech))) 

bars = plt.barh(top_tech.index, top_tech.values, color=colors)

for bar in bars:
    width = bar.get_width()
    plt.text(width + 1, bar.get_y() + bar.get_height()/2, 
             f'{width:.1f}%', va='center', fontsize=10, fontweight='bold')

plt.title(f'Top Technologies for Python Developers (Total jobs: {total_vacancies})', fontsize=14, pad=20)
plt.xlabel('Percentage of Vacancies (%)', fontsize=12)
plt.xlim(0, max(top_tech.values) + 10) 
plt.grid(axis='x', linestyle='--', alpha=0.6)
plt.tight_layout()
plt.savefig('general_tech_ranking.png')
plt.show()